In [ ]:
import os
import numpy as np
import scipy as sp
import matplotlib as mpl
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

In [ ]:
df_path= '/spot_count_dataframes'
dfs_to_load = [f for f in os.listdir(df_path) if not f.startswith('.')]
dfs_to_load

In [ ]:
#Read in spot count dataframes
final_df=pd.DataFrame(columns=['cell_id', 'cell_area', 'nb_rna', 'mean_intensity', 'median_intensity', 'solidity', 'Replicate',
       'eccentricity', 'Position_Name', 'Plasmid', 'Media', 'Time', 'Strain'])

In [ ]:
for df in dfs_to_load:
    data = pd.read_csv(os.path.join(df_path, df), index_col=0)
    final_df= pd.concat([final_df, data], ignore_index=True)

In [ ]:
final_df

In [ ]:
#Read in spot intensity dataframes
spotint_path= '/spot_intensity_dataframes'
spotint_to_load = [f for f in os.listdir(spotint_path) if not f.startswith('.')]
spotint_df=pd.DataFrame(columns=['image', 'spot_intensity'])
for df in spotint_to_load:
    data = pd.read_csv(os.path.join(spotint_path, df), index_col=0)
    spotint_df= pd.concat([spotint_df, data], ignore_index=True)

mapping_dict = final_df.groupby("image")[["Replicate", "Plasmid", "Media", "Strain", "Time"]].first().to_dict("index")

# Map "image" values in spot_int_df to assign the columns
spotint_df[["Replicate", "Plasmid", "Media", "Strain", "Time"]] = spotint_df["image"].map(mapping_dict).apply(pd.Series)

spotint_df['Strain_Plasmid']=spotint_df['Strain']+'_'+spotint_df['Plasmid']
print(spotint_df)

In [ ]:
##Dropping the rare saturated spots
condition = (spotint_df['spot_intensity'] >= 65535)
# Drop rows based on the condition
spotint_df = spotint_df[~condition]
spotint_df

In [ ]:
#Change nan in nb_rna to zeros - these are rows where no spots were detected in the image
final_df['nb_rna'] = final_df['nb_rna'].fillna(0)
final_df

In [ ]:
#Plotting and filtering based on area, eccentricity, solidity
#solidity
#plt.figure(figsize=(5,2.5))
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(6, 10))
ax1 = sns.histplot(data=final_df, x='eccentricity', log_scale=False, stat='probability', common_norm=False, bins = 30, kde = True, hue = 'Replicate', legend = True, ax=ax1)
eccentric_line = [0.6] 
for line in eccentric_line:
    ax1.axvline(x=line, color='r', linestyle='--')
ax2 = sns.histplot(data=final_df, x='cell_area', log_scale=False, stat='probability', common_norm=False, bins = 30, kde = True, hue = 'Replicate', legend = True, ax=ax2)
vertical_lines = [50, 250] 
for line in vertical_lines:
    ax2.axvline(x=line, color='r', linestyle='--')
ax3 = sns.histplot(data=final_df, x='solidity', log_scale=False, stat='probability', common_norm=False, bins = 30, kde = True, hue = 'Replicate', legend = True, ax=ax3)
vertical_lines = [0.8] 
for line in vertical_lines:
    ax3.axvline(x=line, color='r', linestyle='--')
plt.show()

#plt.savefig('/FISHFiltering_Plot.pdf')


#Filtering - Makes a new df for each step, and one for all the steps, prints number of labels removed, this is sequential, but final filter is additive
area_df = final_df[(final_df['cell_area']>= 50) & (final_df['cell_area']<= 250)]
area_lines_removed = len(final_df) - len(area_df)

sol_df = area_df[(area_df['solidity'] >= 0.8)]
sol_lines_removed = len(area_df) - len(sol_df)

ecc_df = sol_df[(sol_df['eccentricity'] >= 0.6)]
ecc_lines_removed = len(sol_df) - len(ecc_df)

filtered_df = final_df[(final_df['cell_area']>= 50) & (final_df['cell_area']<= 250) & (final_df['eccentricity'] >= 0.6) & (final_df['solidity'] >= 0.8)]

all_lines_removed_additive = len(final_df) - len(filtered_df)
all_lines_removed_sequential = len(final_df)-len(ecc_df)
removed_df = final_df[~final_df.index.isin(filtered_df.index)]
print("Unfiltered:", len(final_df))
print("Number of labels removed from area value:", area_lines_removed)
print("Number of labels removed from sol value:", sol_lines_removed)
print("Number of labels removed from ecc value:", ecc_lines_removed)
print("Number of labels removed with additive filters:", all_lines_removed_additive)
print("Number of labels removed with sequential filters:", all_lines_removed_sequential)

In [ ]:
#Find and remove any saturated cells from csvs created in bigfish code
rna_sat_path= '/rna_saturation'

rna_sat_to_load = [f for f in os.listdir(rna_sat_path) if not f.startswith('.')]
rna_saturatedcells_df=pd.DataFrame(columns=['Position_Name', 'cell_id'])

for df in rna_sat_to_load:
    data = pd.read_csv(os.path.join(rna_sat_path, df), index_col=0)
    rna_saturatedcells_df= pd.concat([rna_saturatedcells_df, data], ignore_index=True)

rna_saturatedcells_df



In [ ]:
# Convert sat_df into a set of tuples for fast lookup
sat_set = set(zip(rna_saturatedcells_df['Position_Name'], rna_saturatedcells_df['cell_id']))

# Filter out rows where (A, B) exist in sat_set
filtered_df_nosat = filtered_df[~filtered_df[['Position_Name', 'cell_id']].apply(tuple, axis=1).isin(sat_set)]

# Result
print(f"Original final_df length: {len(filtered_df)}")
print(f"Labels to remove: {len(rna_saturatedcells_df)}")
print(f"df after removal: {len(filtered_df_nosat)}")


In [ ]:
#Identify number of cells per group to downsample
keepstrains = ['WT', 'ssaL_KO', 'ssrB_KO']
keepplasmid = ['pLL1033', 'No_pLL', 'pLL829'] #pLL1033 is the PssaG-sfGFP(LVA) reporter, pLL829 is the P-B0034 ssrB overexpression plasmid
keeptime = [4]
dftosamp = filtered_df_nosat[filtered_df_nosat['Strain'].isin (keepstrains)]
groups = dftosamp.groupby(['Plasmid', 'Strain', 'Replicate', 'Time', 'Media']).size().reset_index(name='Count')
print(groups)

In [ ]:
# Define the number of samples per group
samples_per_group = 300

# Function to sample each group
def sample_group(group):
    return group.sample(samples_per_group)

# Apply sampling to each group and concatenate the results
plotting_df = dftosamp.groupby(['Plasmid', 'Strain', 'Replicate', 'Time', 'Media'], group_keys=False).apply(sample_group)
groups = plotting_df.groupby(['Plasmid', 'Strain', 'Replicate', 'Time', 'Media']).size().reset_index(name='Count')
print(groups)

In [ ]:
pos_cells=plotting_df.groupby(['Strain', 'Plasmid', 'Media', 'Replicate'])['nb_rna'].agg(lambda c: (c>0).sum()/len(c)).reset_index()
pos_cells.rename(columns={"nb_rna": "frac_cells_spot_positive"}, inplace=True)
pos_cells['Strain_Plasmid'] = pos_cells['Strain'] + '_' + pos_cells['Plasmid']
pos_cells['Strain_Plasmid_Media'] = pos_cells['Strain'] + '_' + pos_cells['Plasmid'] + '_' + pos_cells['Media']

pos_cells

In [ ]:
strain_plasmid_media = ['ssaL_KO_No_pLL_MGM', 'ssaL_KO_No_pLL_M9', 'WT_No_pLL_MGM', 'WT_No_pLL_M9']
fig = plt.figure(figsize=(4, 2.5))

plot_df = pos_cells[pos_cells['Strain_Plasmid_Media'].isin(strain_plasmid_media)]
order = ['WT_No_pLL_MGM', 'WT_No_pLL_M9','ssaL_KO_No_pLL_MGM', 'ssaL_KO_No_pLL_M9']
sns.set_style('ticks') #can change this to darkgrid or other styles if you want different background/lines
palette = {'ssaL_KO_No_pLL_MGM':'black', 'ssaL_KO_No_pLL_M9':'grey', 'WT_No_pLL_MGM':'limegreen', 'WT_No_pLL_M9':'grey' }
sns.swarmplot(data=plot_df, x='Strain_Plasmid_Media', y='frac_cells_spot_positive', hue = 'Strain_Plasmid_Media', edgecolor = 'black', linewidth = 0.75, order=order, palette=palette)
snsplot = sns.barplot(data=plot_df, x='Strain_Plasmid_Media', y='frac_cells_spot_positive', hue = 'Strain_Plasmid_Media', errorbar='sd', order=order, palette=palette)
plt.ylim(-0.05, 1)
sns.despine()
#plt.savefig('/MainFig1_GFP+.pdf')

In [ ]:
strain_plasmid_media = ['ssaL_KO_No_pLL_MGM', 'ssaL_KO_No_pLL_M9', 'WT_No_pLL_MGM', 'WT_No_pLL_M9', 'WT_pLL1033_MGM', 'WT_pLL1033_M9', 'ssrB_KO_pLL829_MGM', 'ssrB_KO_pLL829_M9', 'ssrB_KO_No_pLL_MGM', 'ssrB_KO_No_pLL_M9']
fig = plt.figure(figsize=(10, 5))

plot_df = pos_cells[pos_cells['Strain_Plasmid_Media'].isin(strain_plasmid_media)]
order = ['ssaL_KO_No_pLL_MGM', 'ssrB_KO_No_pLL_MGM', 'WT_No_pLL_MGM', 'WT_pLL1033_MGM', 'ssrB_KO_pLL829_MGM',
        'ssaL_KO_No_pLL_M9', 'ssrB_KO_No_pLL_M9', 'WT_No_pLL_M9', 'WT_pLL1033_M9', 'ssrB_KO_pLL829_M9',]
sns.set_style('ticks') #can change this to darkgrid or other styles if you want different background/lines
palette = {'ssaL_KO_No_pLL_MGM':'black', 'ssaL_KO_No_pLL_M9':'black', 'ssrB_KO_No_pLL_MGM': 'steelblue', 'ssrB_KO_No_pLL_M9':'steelblue', 'WT_No_pLL_MGM':'limegreen', 'WT_No_pLL_M9':'limegreen',
          'WT_pLL1033_MGM':'green', 'WT_pLL1033_M9':'green', 'ssrB_KO_pLL829_MGM':'rebeccapurple', 'ssrB_KO_pLL829_M9':'rebeccapurple'}
sns.swarmplot(data=plot_df, x='Strain_Plasmid_Media', y='frac_cells_spot_positive', hue = 'Strain_Plasmid_Media', edgecolor = 'black', linewidth = 0.75, order=order, palette=palette)
snsplot = sns.barplot(data=plot_df, x='Strain_Plasmid_Media', y='frac_cells_spot_positive', hue = 'Strain_Plasmid_Media', errorbar='sd', order=order, palette=palette)
plt.ylim(-0.05, 1)
sns.despine()
#plt.savefig('/SuppFig1_GFP+.pdf')

In [ ]:
# List to store the names of the dynamically created DataFrames
df_names = []

# Assuming df is your original DataFrame
for (strain, media, plasmid), group_df in plotting_df.groupby(['Strain', 'Media', 'Plasmid']):
    # Create a variable name dynamically for each group
    df_name = f'df_{strain}_{media}_{plasmid}'
    globals()[df_name] = group_df
    df_names.append(df_name)  # Add the DataFrame name to the list

# Print names and lengths of all the dynamically created DataFrames
for name in df_names:
    obj = globals()[name]
    print(f'{name}: {len(obj)} rows')


In [ ]:
# Dictionary to store the new dataframes
sorted_counts_dict = {}

for name in df_names:
    df = globals()[name]  # Retrieve the DataFrame from globals()
    
    counts_df = df.loc[:, 'nb_rna'] 
    counts_sorted_df = np.sort(counts_df)
    
    p_df = 1. * np.arange(len(counts_df)) / (len(df) - 1)
    
    # Store results in a dictionary
    sorted_counts_dict[f'counts_{name}'] = counts_sorted_df
    sorted_counts_dict[f'p_{name}'] = p_df

# Example of accessing one of the generated arrays:
for key, value in sorted_counts_dict.items():
    print(f"{key}: {value[:5]}")  


In [ ]:
selected_dfs = ['df_ssaL_KO_MGM_No_pLL', 'df_ssaL_KO_M9_No_pLL', 'df_WT_MGM_No_pLL', 'df_WT_M9_No_pLL']  # Replace with actual names
color_dict = {'df_ssaL_KO_MGM_No_pLL':'black', 'df_ssaL_KO_M9_No_pLL':'grey', 'df_WT_MGM_No_pLL':'limegreen', 'df_WT_M9_No_pLL':'grey'}

# Create a figure
fig, ax1 = plt.subplots(figsize=(8, 6))

# Loop through selected datasets and plot
for name in selected_dfs:
    if f'counts_{name}' in sorted_counts_dict and f'p_{name}' in sorted_counts_dict:
        counts_sorted_df = sorted_counts_dict[f'counts_{name}']
        p_df = sorted_counts_dict[f'p_{name}']
        color = color_dict.get(name, 'black')
        ax1.plot(counts_sorted_df, p_df, label=name, linewidth=2, color=color)

# Customize the plot
ax1.set_xlabel('Count')
ax1.set_ylabel('CDF')
ax1.set_xlim(-1, 20) #note there are some very rare outliers depending on downsample that can make the distributions of the bulk of the population hard to see - line extending past axis indicates this, can remove limits to see what the outlier is
ax1.legend()
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
#plt.savefig('/MainFig1_CDF_noxlim.pdf')

In [ ]:
selected_dfs = ['df_ssaL_KO_MGM_No_pLL', 'df_ssrB_KO_MGM_No_pLL', 'df_WT_MGM_No_pLL', 'df_WT_MGM_pLL1033', 'df_ssrB_KO_MGM_pLL829']  # Replace with actual names
color_dict = {'df_ssaL_KO_MGM_No_pLL':'black', 'df_ssrB_KO_MGM_No_pLL':'steelblue', 'df_WT_MGM_No_pLL':'limegreen', 'df_WT_MGM_pLL1033':'green', 'df_ssrB_KO_MGM_pLL829':'rebeccapurple' }

# Create a figure
fig, ax1 = plt.subplots(figsize=(8, 6))

# Loop through selected datasets and plot
for name in selected_dfs:
    if f'counts_{name}' in sorted_counts_dict and f'p_{name}' in sorted_counts_dict:
        counts_sorted_df = sorted_counts_dict[f'counts_{name}']
        p_df = sorted_counts_dict[f'p_{name}']
        color = color_dict.get(name, 'black')
        ax1.plot(counts_sorted_df, p_df, label=name, linewidth=2, color=color)

# Customize the plot
ax1.set_xlabel('Count')
ax1.set_ylabel('CDF')
ax1.set_xlim(-1, 20) #note there are some very rare outliers depending on downsample that can make the distributions of the bulk of the population hard to see - line extending past axis indicates this, can remove limits to see what the outlier is
ax1.legend()
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

#plt.savefig('/SuppFig1_CDF_noxlim_MGM.pdf')

In [ ]:
selected_dfs = ['df_ssaL_KO_M9_No_pLL', 'df_ssrB_KO_M9_No_pLL', 'df_WT_M9_No_pLL', 'df_WT_M9_pLL1033', 'df_ssrB_KO_M9_pLL829']  # Replace with actual names
color_dict = {'df_ssaL_KO_M9_No_pLL':'black', 'df_ssrB_KO_M9_No_pLL':'steelblue', 'df_WT_M9_No_pLL':'limegreen', 'df_WT_M9_pLL1033':'green', 'df_ssrB_KO_M9_pLL829':'rebeccapurple' }

# Create a figure
fig, ax1 = plt.subplots(figsize=(8, 6))

# Loop through selected datasets and plot
for name in selected_dfs:
    if f'counts_{name}' in sorted_counts_dict and f'p_{name}' in sorted_counts_dict:
        counts_sorted_df = sorted_counts_dict[f'counts_{name}']
        p_df = sorted_counts_dict[f'p_{name}']
        color = color_dict.get(name, 'black')
        ax1.plot(counts_sorted_df, p_df, label=name, linewidth=2, color=color)

# Customize the plot
ax1.set_xlabel('Count')
ax1.set_ylabel('CDF')
ax1.set_xlim(-1, 20) #note there are some very rare outliers that can make it hard to see distributions depending on downsample - line extending past axis indicates this, can remove limits to see what the outlier is
ax1.legend()
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

#plt.savefig('/SuppFig1_CDF_noxlim_M9.pdf')

In [ ]:
#identifying GFP+ threshold using strains with no reporter. 
plasmid = ['No_pLL']
media = ['MGM']
strain = ['WT']
df_WT = plotting_df[plotting_df['Plasmid'].isin(plasmid) & plotting_df['Media'].isin(media) & plotting_df['Strain'].isin(strain)] 


GFP_wt_std = df_WT['median_intensity'].std()
GFP_wt_mean = df_WT['median_intensity'].mean()
print(GFP_wt_mean)
print(GFP_wt_std)
#plt.figure(figsize=(5,2.5))
ax = sns.histplot(data=df_WT, x='median_intensity', log_scale=True, stat='probability', common_norm=False, bins = 50, kde = True, hue='Replicate', legend = True)
vertical_lines = [GFP_wt_mean, 2*GFP_wt_mean ]
for line in vertical_lines:
    plt.axvline(x=line, color='r', linestyle='--')

In [ ]:
#Pull out just the reporter strain and plot intensities
plasmid = ['pLL1033']
media = ['MGM']
GFP_threshold = 2*GFP_wt_mean
FISH_threshold = 0
df_1033_MGM = plotting_df[plotting_df['Plasmid'].isin(plasmid) & plotting_df['Media'].isin(media)] 
df_1033_MGM['GFP_Pos'] = df_1033_MGM['median_intensity'] > GFP_threshold
df_1033_MGM['Transc_Pos'] = df_1033_MGM['nb_rna'] > FISH_threshold

ax = sns.kdeplot(data=df_1033_MGM, x='median_intensity', log_scale=True, common_norm=False, hue='Replicate', legend = True, palette = 'viridis')
vertical_lines = [GFP_wt_mean, 2*GFP_wt_mean ]
for line in vertical_lines:
    plt.axvline(x=line, color='black', linestyle='--')
sns.despine()

In [ ]:
#Plot grid of reporter and/or transcript positive percentages
df_1033_MGM['GFP_Pos'] = df_1033_MGM['median_intensity'] > GFP_threshold
df_1033_MGM['Transc_Pos'] = df_1033_MGM['nb_rna'] > FISH_threshold

group_counts = df_1033_MGM.groupby(['GFP_Pos', 'Transc_Pos']).size().reset_index(name='count')
total_count = group_counts['count'].sum()  # Calculate the total count
group_counts['percentage'] = (group_counts['count'] / total_count) * 100 

gfp_order = [False, True]
transc_order = [True, False]
# Convert the 'Category' column to a categorical type with the specified order
group_counts['GFP_Pos'] = pd.Categorical(group_counts['GFP_Pos'], categories=gfp_order, ordered=True)
group_counts['Transc_Pos'] = pd.Categorical(group_counts['Transc_Pos'], categories=transc_order, ordered=True)


pivot = group_counts.pivot(index="Transc_Pos", columns="GFP_Pos", values="percentage")
sns.heatmap(pivot, annot = True, fmt=".0f", cmap = 'viridis', vmax =50, vmin = 0)
#plt.savefig('/transcriptrep_heatmap.pdf')

In [ ]:
#Plot reporter distribution per spot count
vertical_lines = [2*GFP_wt_mean]

# Define the values of nb_rna you want to plot
selected_values = [0, 1, 2, 3, 4]  
combined_threshold = 5  # Threshold above which values will be combined

# Create the additional subset for combined values
combined_subset = df_1033_MGM[df_1033_MGM['nb_rna'] >= combined_threshold]

# Create subplots: one row per selected nb_rna value + 1 for combined values
fig, axes = plt.subplots(len(selected_values) + 1, 1, figsize=(8, 2 * (len(selected_values) + 1)), sharex=True)

# Ensure axes is iterable even if there's only one subplot
if len(selected_values) == 1:
    axes = [axes]

# Loop through selected values and plot each KDE in a separate subplot
for ax, value in zip(axes, selected_values):
    subset = df_1033_MGM[df_1033_MGM['nb_rna'] == value]  
    if not subset.empty:  # Check if data exists
        sns.kdeplot(subset['median_intensity'], ax=ax, linewidth=2, log_scale=True, fill=True, color='limegreen')
        ax.set_title(f'nb_rna {value}')  # Set subplot title
        ax.set_ylabel('Density')  # Set y-axis label
        for line in vertical_lines:
            ax.axvline(x=line, color='black', linestyle='--')

        # Add text with count in the top-right corner
        count = len(subset)
        ax.text(0.95, 0.95, f'n={count}', transform=ax.transAxes,
                verticalalignment='top', horizontalalignment='right',
                fontsize=10, bbox=dict(facecolor='white', alpha=0.6, edgecolor='white'))

# Plot the combined KDE for values above the threshold
ax_combined = axes[-1]  # Last subplot
if not combined_subset.empty:
    sns.kdeplot(combined_subset['median_intensity'], ax=ax_combined, linewidth=2, log_scale=True, fill=True, color='limegreen')
    ax_combined.set_title(f'nb_rna ≥ {combined_threshold}')
    ax_combined.set_ylabel('Density')
    for line in vertical_lines:
        ax_combined.axvline(x=line, color='black', linestyle='--')

    # Add text with count in the top-right corner
    count = len(combined_subset)
    ax_combined.text(0.95, 0.95, f'n={count}', transform=ax_combined.transAxes,
                     verticalalignment='top', horizontalalignment='right',
                     fontsize=10, bbox=dict(facecolor='white', alpha=0.6, edgecolor='white'))

# Set common x-axis label
ax_combined.set_xlabel('Median Intensity')

plt.tight_layout()


plt.tight_layout()
#plt.savefig("/transcriptnumber_repint_kde.pdf')


In [ ]:

plasmid = ['No_pLL']
media = ['MGM', 'M9']
strain = ['WT']
df_int_plot = spotint_df[spotint_df['Plasmid'].isin(plasmid) & spotint_df['Media'].isin(media) & spotint_df['Strain'].isin(strain)]
palette = {'MGM':'limegreen', 'M9':'grey'}
fig, ax = plt.subplots(figsize=(8, 6))
sns.histplot(
    data=df_int_plot, x='spot_intensity', hue='Media', palette=palette, log_scale=True,
    stat='probability', bins=35, common_norm=False, ax=ax, edgecolor="white"
)
for i, media_type in enumerate(df_int_plot['Media'].unique()):
    count = len(df_int_plot[df_int_plot['Media'] == media_type])
    ax.text(
        0.95, 0.95 - (i * 0.05), f'{media_type}: n={count}', 
        transform=ax.transAxes, verticalalignment='top', horizontalalignment='right',
        fontsize=10
    )
#ax.set_xlim(200, 100000)

sns.despine()
#plt.savefig('/spotint_main_hist.pdf')

In [ ]:
#Spot intensity for all strains in MGM
plasmid = ['No_pLL', 'pLL829', 'pLL1033']
media = ['MGM']
strain = ['WT', 'ssrB_KO', 'ssaL_KO']
df_int_plot = spotint_df[spotint_df['Plasmid'].isin(plasmid) & spotint_df['Media'].isin(media) & spotint_df['Strain'].isin(strain)]
palette = {'ssaL_KO_No_pLL':'black', 'ssrB_KO_No_pLL':'steelblue', 'WT_No_pLL':'limegreen', 'WT_pLL1033':'green', 'ssrB_KO_pLL829':'rebeccapurple' }

fig, ax = plt.subplots(figsize=(8, 6))
sns.kdeplot(
    data=df_int_plot, x='spot_intensity', palette = palette, hue='Strain_Plasmid', log_scale=True, 
    common_norm=False, ax=ax, legend = False, fill = True
)
for i, media_type in enumerate(df_int_plot['Strain_Plasmid'].unique()):
    count = len(df_int_plot[df_int_plot['Strain_Plasmid'] == media_type])
    ax.text(
        0.95, 0.95 - (i * 0.05), f'{media_type}: n={count}', 
        transform=ax.transAxes, verticalalignment='top', horizontalalignment='right',
        fontsize=10)
#ax.set_xlim(100, 100000)

sns.despine()
#plt.savefig('/spotint_supp_kde_MGM.pdf')

In [ ]:
plasmid = ['No_pLL', 'pLL829', 'pLL1033']
media = ['M9']
strain = ['WT', 'ssrB_KO', 'ssaL_KO']
df_int_plot = spotint_df[spotint_df['Plasmid'].isin(plasmid) & spotint_df['Media'].isin(media) & spotint_df['Strain'].isin(strain)]
palette = {'ssaL_KO_No_pLL':'black', 'ssrB_KO_No_pLL':'steelblue', 'WT_No_pLL':'limegreen', 'WT_pLL1033':'green', 'ssrB_KO_pLL829':'rebeccapurple' }

fig, ax = plt.subplots(figsize=(8, 6))
sns.kdeplot(
    data=df_int_plot, x='spot_intensity', hue='Strain_Plasmid', log_scale=True,
    common_norm=False, ax=ax, legend = False, fill = True, palette = palette
)
for i, media_type in enumerate(df_int_plot['Strain_Plasmid'].unique()):
    count = len(df_int_plot[df_int_plot['Strain_Plasmid'] == media_type])
    ax.text(
        0.95, 0.95 - (i * 0.05), f'{media_type}: n={count}', 
        transform=ax.transAxes, verticalalignment='top', horizontalalignment='right',
        fontsize=10)

#ax.set_xlim(100, 100000)

sns.despine()
#plt.savefig('/spotint_supp_kde_M9.pdf')